**log1p 변환**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

In [2]:
train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/smart_factory/data/raw/train.csv')
test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/smart_factory/data/raw/test.csv')

In [3]:
drop_cols = ['ID', 'scenario_id', 'avg_delay_minutes_next_30m', 'layout_id']
target = 'avg_delay_minutes_next_30m'

In [4]:
X      = train.drop(columns=drop_cols)
y_raw  = train[target]
y      = np.log1p(y_raw)          ##### log1p 변환
X_test = test.drop(columns=['ID', 'scenario_id', 'layout_id'])

In [5]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_pred  = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
mae_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(
        objective='mae',
        metric='mae',
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    val_pred_log = model.predict(X_val)
    val_pred     = np.expm1(val_pred_log)       # ✅ 역변환
    oof_pred[val_idx] = val_pred

    fold_mae = mean_absolute_error(y_raw.iloc[val_idx], val_pred)   # ✅ 원본 스케일 MAE
    mae_scores.append(fold_mae)
    print(f"Fold {fold} MAE: {fold_mae:.4f}")

    test_pred += np.expm1(model.predict(X_test)) / N_FOLDS          # ✅ 역변환 후 평균

print(f"\n✅ 각 Fold MAE: {[round(s, 4) for s in mae_scores]}")
print(f"✅ 평균 MAE:    {np.mean(mae_scores):.4f}")
print(f"✅ OOF MAE:     {mean_absolute_error(y_raw, oof_pred):.4f}")  # 가장 신뢰도 높음


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.243391 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19688
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 90
[LightGBM] [Info] Start training from score 2.307431
[200]	valid_0's l1: 0.490442
[400]	valid_0's l1: 0.483169
[600]	valid_0's l1: 0.477907
[800]	valid_0's l1: 0.473495
[1000]	valid_0's l1: 0.47017
Fold 1 MAE: 8.7605
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.222076 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19686
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 90
[LightGBM] [Info] Start training from score 2.306460
[200]	valid_0's l1: 0.493942
[400]	valid_0's l1: 0.487462
[600]	valid_0's l1: 0.481655
[800]	valid_0's l1: 0.477245
[1000]	valid_0's l1: 0.473513
Fo

In [6]:
submission = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/smart_factory/sample_submission.csv')
submission['avg_delay_minutes_next_30m'] = test_pred
submission.to_csv('submission_v2_kfold_log1p.csv', index=False)
print("\nsubmission_v2_kfold_log1p.csv 저장")



submission_v2_kfold_log1p.csv 저장
